In [1]:
import sys

import pm4py

import pandas as pd
import numpy as np

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts
from model.next_event_model import ProcessLSTM, train_ProcessLSTM, validate_ProcessLSTM

### --- Preprocess dataset ---

In [2]:
set_seed(seed=42)

In [3]:
log = pm4py.read_xes("../../data/RequestForPayment.xes")

C:\Users\dcoralage\Downloads\counterfactual_prediction_experiments\counterfactual_env\lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
C:\Users\dcoralage\Downloads\counterfactual_prediction_experiments\counterfactual_env\lib\site-packages\pm4py\util\dt_parsing\parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/6886 [00:00<?, ?it/s]

In [4]:
df = pm4py.convert_to_dataframe(log)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36796 entries, 0 to 36795
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   id                         36796 non-null  object             
 1   org:resource               36796 non-null  object             
 2   concept:name               36796 non-null  object             
 3   time:timestamp             36796 non-null  datetime64[ns, UTC]
 4   org:role                   36796 non-null  object             
 5   case:Rfp_id                36796 non-null  object             
 6   case:Project               36796 non-null  object             
 7   case:Task                  36796 non-null  object             
 8   case:concept:name          36796 non-null  object             
 9   case:OrganizationalEntity  36796 non-null  object             
 10  case:Cost Type             36796 non-null  int64              
 11  ca

In [6]:
df.isnull().any()

id                           False
org:resource                 False
concept:name                 False
time:timestamp               False
org:role                     False
case:Rfp_id                  False
case:Project                 False
case:Task                    False
case:concept:name            False
case:OrganizationalEntity    False
case:Cost Type               False
case:RequestedAmount         False
case:Activity                False
case:RfpNumber               False
dtype: bool

In [7]:
df = df.drop(columns=['case:Rfp_id', 'case:RfpNumber', 'case:Cost Type', 'id', 'case:Project', 'case:Task'])

In [8]:
df['case:concept:name'] = df['case:concept:name'].astype('string')
df['concept:name'] = df['concept:name'].astype('string')
df['org:resource'] = df['org:resource'].astype('string')
df['org:role'] = df['org:role'].astype('string')
df['case:Activity'] = df['case:Activity'].astype('string')
df['case:OrganizationalEntity'] = df['case:OrganizationalEntity'].astype('string')

df['case:RequestedAmount'] = df['case:RequestedAmount'].astype(np.float32)

df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], errors='coerce')

In [9]:
df = df.sort_values(by=['case:concept:name', 'time:timestamp'], ascending=[True, True])

In [10]:
df['time_delta'] = df.groupby('case:concept:name')['time:timestamp'].diff()
df['time_delta'] = df['time_delta'].dt.total_seconds().astype(np.float32)
df['time_delta'] = df['time_delta'].fillna(0)

In [11]:
exclude_cols = ["case:concept:name", "time:timestamp"]

sorted_cols = sorted(
    [c for c in df.columns if c not in exclude_cols]
)

df = df[exclude_cols + sorted_cols]

In [12]:
# Remove activities only found in validation data
df = df[df['concept:name'] != 'Request For Payment FOR_APPROVAL by SUPERVISOR']

In [13]:
df.head(20)

,case:concept:name,time:timestamp,case:Activity,case:OrganizationalEntity,case:RequestedAmount,concept:name,org:resource,org:role,time_delta
389,request for payment 147529,2017-02-14 15:34:34+00:00,UNKNOWN,organizational unit 65458,137.526306,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
390,request for payment 147529,2017-02-14 15:34:43+00:00,UNKNOWN,organizational unit 65458,137.526306,Request For Payment FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,9.0
391,request for payment 147529,2017-02-15 14:48:02+00:00,UNKNOWN,organizational unit 65458,137.526306,Request Payment,SYSTEM,UNDEFINED,83599.0
392,request for payment 147529,2017-02-20 17:32:08+00:00,UNKNOWN,organizational unit 65458,137.526306,Payment Handled,SYSTEM,UNDEFINED,441846.0
601,request for payment 147534,2017-03-02 15:55:43+00:00,UNKNOWN,organizational unit 65463,59.567024,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
602,request for payment 147534,2017-03-02 15:58:27+00:00,UNKNOWN,organizational unit 65463,59.567024,Request For Payment APPROVED by PRE_APPROVER,STAFF MEMBER,PRE_APPROVER,164.0
603,request for payment 147534,2017-03-02 16:07:38+00:00,UNKNOWN,organizational unit 65463,59.567024,Request For Payment FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,551.0
604,request for payment 147534,2017-03-06 13:57:31+00:00,UNKNOWN,organizational unit 65463,59.567024,Request Payment,SYSTEM,UNDEFINED,337793.0
605,request for payment 147534,2017-03-13 17:31:05+00:00,UNKNOWN,organizational unit 65463,59.567024,Payment Handled,SYSTEM,UNDEFINED,617614.0
660,request for payment 147539,2017-03-06 14:40:07+00:00,UNKNOWN,organizational unit 65458,47.927757,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0


In [14]:
num_cases = df['case:concept:name'].nunique()
print(f"Total number of unique cases: {num_cases}")

Total number of unique cases: 6886


### --- Feature Configurations ---

In [15]:
# --- Define feature specs ---
feature_specs = {

    "time_delta": {
        "type":           "continuous",
        "level":          "event",
        "vary":           True,
        "quantile_low":   0.20,
        "quantile_high":  0.80, 
    },

    "case:RequestedAmount": {
        "type":           "continuous",
        "level":          "case",
        "vary":           True,
        "quantile_low":   0.05,
        "quantile_high":  0.90, 
    },

    "org:resource": {
        "type":           "categorical",
        "level":          "event",
        "vary":           True,
    },

    "org:role": {
        "type":           "categorical",
        "level":          "event",
        "vary":           True,
    },
    
    "case:Activity": {
        "type":           "categorical",
        "level":          "case",
        "vary":           True,
    },

    "case:OrganizationalEntity": {
        "type":           "categorical",
        "level":          "case",
        "vary":           True,
    },

    # immutable
    "concept:name": {
        "type":           "categorical", 
        "level":          "event",
        "vary":           False
    },
}

In [16]:
feature_config = FeatureConfig.from_dataframe(
    df=df,
    feature_specs=feature_specs,
    activity_feature="concept:name",
    is_robust=True,
    default_quantile_low=0.05,
    default_quantile_high=0.95
)

feature_config.save()

In [17]:
# feature_config = FeatureConfig.load()

In [18]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:Activity', 'case:OrganizationalEntity', 'case:RequestedAmount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [3.00, 325445.40]                        57230.0000 quantile_derived    
case:RequestedAmount           continuous     case     yes    [10.54, 665.70]                          74.7009    quantile_derived    
org:resource                   categorical    event    yes    ['STAFF MEMBER', 'SYSTEM']               N/A        data_derived        
or

### --- Next event prediction model ---

In [19]:
# Transform nan cols to NA
cat_cols = df.select_dtypes(include=["string"]).columns
for col in cat_cols:
    df[col] = df[col].fillna("NA").astype('string')

In [20]:
case_ids = df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = df[df["case:concept:name"].isin(train_cases)].copy()
val_df   = df[df["case:concept:name"].isin(val_cases)].copy()

In [21]:
preprocessor_artifacts = PreprocessorArtifacts.build(
    df=train_df,
    feature_config=feature_config,      
    scaler_type="robust",
)

preprocessor_artifacts.save()

In [22]:
# preprocessor_artifacts = PreprocessorArtifacts.load()

In [23]:
preprocessor_artifacts.summary()

===================PreprocessorArtifacts====================
  scaler:              RobustScaler
  encoders:            ['org:resource', 'org:role', 'case:Activity', 'case:OrganizationalEntity', 'concept:name']
  activity_prototypes: 18 activities


In [24]:
# Transform nan cols to 0
float_cols = df.select_dtypes(include=["float32", "float64"]).columns
df[float_cols] = df[float_cols].fillna(0)
train_df[float_cols] = train_df[float_cols].fillna(0)
val_df[float_cols] = val_df[float_cols].fillna(0)

In [25]:
train_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    df=train_df,
    case_id_field="case:concept:name", 
    sort_field="time:timestamp"
)

val_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    df=val_df,
    case_id_field="case:concept:name", 
    sort_field="time:timestamp"
)

In [26]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [27]:
print(preprocessor_artifacts.get_categorical_feature_cardinality())

{'dynamic_categorical_info': {'concept:name': 18, 'org:resource': 2, 'org:role': 8}, 'static_categorical_info': {'case:Activity': 6, 'case:OrganizationalEntity': 36}}


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [29]:
device

device(type='cuda')

In [30]:
criterion = torch.nn.CrossEntropyLoss()

In [31]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Rfp-model_output.txt")

Epoch 020/100 | Train Loss: 0.3154 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.3018 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.2880 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.2807 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.2775 | LR: 1.00e-06
Time taken for next event model (training): 608.166763 seconds
Time taken for next event model (validation): 0.471604 seconds
Val loss: {'loss': 0.37001264359225466, 'accuracy': 0.8677533187699547, 'f1_macro': 0.5619599263848283, 'f1_weighted': 0.8467376075320545}


In [32]:
embedding_metadata = preprocessor_artifacts.get_embedding_metadata()

model = ProcessLSTM(
    dynamic_categorical_info=embedding_metadata["dynamic_categorical_info"],
    static_categorical_info=embedding_metadata["static_categorical_info"],
    n_dynamic_continuous=embedding_metadata["n_dynamic_continuous"],
    n_static_continuous=embedding_metadata["n_static_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ProcessLSTM(
    model=model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

model.save()

In [33]:
# model = ProcessLSTM.load()

In [34]:
val_loss = validate_ProcessLSTM(
    model=model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [35]:
# --- Save processed df ---
if df["time:timestamp"].dt.tz is not None:
    df["time:timestamp"] = df["time:timestamp"].dt.tz_convert(None)
df.to_excel("../../data/bpic20_Rfp.xlsx", index=False, engine="openpyxl")

In [36]:
sys.stdout = original_stdout
log_file.close()